In [ ]:
from __future__ import print_function
import argparse
import inspect
import os
import pickle
import random
import shutil
import sys
import time
import math
from collections import OrderedDict
import traceback
from sklearn.metrics import confusion_matrix
import csv
import numpy as np
import glob
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from tensorboardX import SummaryWriter
from tqdm import tqdm
from feeders.feeder_ntu import Feeder
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

import torch_geometric.transforms as T
from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_add_pool
import inspect
from typing import Any, Dict, Optional

import torch.nn.functional as F
from torch import Tensor
from torch.nn import Dropout, Linear, Sequential

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import reset
from torch_geometric.nn.resolver import (
    activation_resolver,
    normalization_resolver,
)
from torch_geometric.typing import Adj
from torch_geometric.utils import to_dense_batch

from mamba_ssm import Mamba, Mamba2
from torch_geometric.utils import degree, sort_edge_index

import wandb

### torch version too old for timm
### https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/layers
def drop_path(x, drop_prob: float = 0., training: bool = False, scale_by_keep: bool = True):
    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks).
    This is the same as the DropConnect impl I created for EfficientNet, etc networks, however,
    the original name is misleading as 'Drop Connect' is a different form of dropout in a separate paper...
    See discussion: https://github.com/tensorflow/tpu/issues/494#issuecomment-532968956 ... I've opted for
    changing the layer and argument names to 'drop path' rather than mix DropConnect as a layer name and use
    'survival rate' as the argument.
    """
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # work with diff dim tensors, not just 2D ConvNets
    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
    if keep_prob > 0.0 and scale_by_keep:
        random_tensor.div_(keep_prob)
    return x * random_tensor


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks).
    """
    def __init__(self, drop_prob=None, scale_by_keep=True):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
        self.scale_by_keep = scale_by_keep

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training, self.scale_by_keep)

### torch version too old for timm
### https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/layers
def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    # Cut & paste from PyTorch official master until it's in a few official releases - RW
    # Method based on https://people.sc.fsu.edu/~jburkardt/presentations/truncated_normal.pdf
    def norm_cdf(x):
        # Computes standard normal cumulative distribution function
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    if (mean < a - 2 * std) or (mean > b + 2 * std):
        warnings.warn("mean is more than 2 std from [a, b] in nn.init.trunc_normal_. "
                      "The distribution of values may be incorrect.",
                      stacklevel=2)

    with torch.no_grad():
        # Values are generated by using a truncated uniform distribution and
        # then using the inverse CDF for the normal distribution.
        # Get upper and lower cdf values
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)

        # Uniformly fill tensor with values from [l, u], then translate to
        # [2l-1, 2u-1].
        tensor.uniform_(2 * l - 1, 2 * u - 1)

        # Use inverse cdf transform for normal distribution to get truncated
        # standard normal
        tensor.erfinv_()

        # Transform to proper mean, std
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)

        # Clamp to ensure it's in the proper range
        tensor.clamp_(min=a, max=b)
        return tensor


def trunc_normal_(tensor, mean=0., std=1., a=-2., b=2.):
    # type: (Tensor, float, float, float, float) -> Tensor
    r"""Fills the input Tensor with values drawn from a truncated
    normal distribution. The values are effectively drawn from the
    normal distribution :math:`\mathcal{N}(\text{mean}, \text{std}^2)`
    with values outside :math:`[a, b]` redrawn until they are within
    the bounds. The method used for generating the random values works
    best when :math:`a \leq \text{mean} \leq b`.
    Args:
        tensor: an n-dimensional `torch.Tensor`
        mean: the mean of the normal distribution
        std: the standard deviation of the normal distribution
        a: the minimum cutoff value
        b: the maximum cutoff value
    Examples:
        >>> w = torch.empty(3, 5)
        >>> nn.init.trunc_normal_(w)
    """
    return _no_grad_trunc_normal_(tensor, mean, std, a, b)




def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod


def conv_branch_init(conv, branches):
    weight = conv.weight
    n = weight.size(0)
    k1 = weight.size(1)
    k2 = weight.size(2)
    nn.init.normal_(weight, 0, math.sqrt(2. / (n * k1 * k2 * branches)))
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def conv_init(conv):
    if conv.weight is not None:
        nn.init.kaiming_normal_(conv.weight, mode='fan_out')
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        if hasattr(m, 'weight'):
            nn.init.kaiming_normal_(m.weight, mode='fan_out')
        if hasattr(m, 'bias') and m.bias is not None and isinstance(m.bias, torch.Tensor):
            nn.init.constant_(m.bias, 0)
    elif classname.find('BatchNorm') != -1:
        if hasattr(m, 'weight') and m.weight is not None:
            m.weight.data.normal_(1.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            m.bias.data.fill_(0)

class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1):
        super(TemporalConv, self).__init__()
        pad = (kernel_size + (kernel_size-1) * (dilation-1) - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(kernel_size, 1),
            padding=(pad, 0),
            stride=(stride, 1),
            dilation=(dilation, 1),
            )

        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x

class MultiScale_TemporalConv(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size=3,
                 stride=1,
                 dilations=[1,2,3,4],
                 residual=False,
                 residual_kernel_size=1):

        super().__init__()
        assert out_channels % (len(dilations) + 2) == 0, '# out channels should be multiples of # branches'

        # Multiple branches of temporal convolution
        self.num_branches = len(dilations) + 2
        branch_channels = out_channels // self.num_branches
        if type(kernel_size) == list:
            assert len(kernel_size) == len(dilations)
        else:
            kernel_size = [kernel_size]*len(dilations)
        # Temporal Convolution branches
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    branch_channels,
                    kernel_size=1,
                    padding=0
                ),
                nn.BatchNorm2d(branch_channels),
                nn.ReLU(inplace=True),
                TemporalConv(
                    branch_channels,
                    branch_channels,
                    kernel_size=ks,
                    stride=stride,
                    dilation=dilation
                ),
            )
            for ks, dilation in zip(kernel_size, dilations)
        ])

        # Additional Max & 1x1 branch
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0),
            nn.BatchNorm2d(branch_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3,1), stride=(stride,1), padding=(1,0)),
            nn.BatchNorm2d(branch_channels)  # 为什么还要加bn
        ))

        self.branches.append(nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0, stride=(stride,1)),
            nn.BatchNorm2d(branch_channels)
        ))

        # Residual connection
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = TemporalConv(in_channels, out_channels, kernel_size=residual_kernel_size, stride=stride)
        # print(len(self.branches))
        # initialize
        self.apply(weights_init)



    def forward(self, x):
        # Input dim: (N,C,T,V)
        res = self.residual(x)
        branch_outs = []
        for tempconv in self.branches:
            out = tempconv(x)
            branch_outs.append(out)

        out = torch.cat(branch_outs, dim=1)
        out += res
        return out

class unit_tcn(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, stride=1):
        super(unit_tcn, self).__init__()
        pad = int((kernel_size - 1) / 2)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(kernel_size, 1), padding=(pad, 0),
                              stride=(stride, 1), groups=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        conv_init(self.conv)
        bn_init(self.bn, 1)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x


class MHSA(nn.Module):
    def __init__(self, dim_in, dim, A, num_heads=6, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0., insert_cls_layer=0, pe=False, num_point=25,
                 outer=True, layer=0,
                 **kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.num_point = num_point
        self.layer = layer



        h1 = A.sum(0)
        h1[h1 != 0] = 1
        h = [None for _ in range(num_point)]
        h[0] = np.eye(num_point)
        h[1] = h1
        self.hops = 0*h[0]
        for i in range(2, num_point):
            h[i] = h[i-1] @ h1.transpose(0, 1)
            h[i][h[i] != 0] = 1

        for i in range(num_point-1, 0, -1):
            if np.any(h[i]-h[i-1]):
                h[i] = h[i] - h[i - 1]
                self.hops += i*h[i]
            else:
                continue

        self.hops = torch.tensor(self.hops).long()
        #
        self.rpe = nn.Parameter(torch.zeros((self.hops.max()+1, dim)))

        self.w1 = nn.Parameter(torch.zeros(num_heads, head_dim))



        A = A.sum(0)
        A[:, :] = 0

        self.outer = nn.Parameter(torch.stack([torch.eye(A.shape[-1]) for _ in range(num_heads)], dim=0), requires_grad=True)

        self.alpha = nn.Parameter(torch.zeros(1), requires_grad=True)

        self.kv = nn.Conv2d(dim_in, dim * 2, 1, bias=qkv_bias)
        self.q = nn.Conv2d(dim_in, dim, 1, bias=qkv_bias)

        self.attn_drop = nn.Dropout(attn_drop)


        self.proj = nn.Conv2d(dim, dim, 1, groups=6)

        self.proj_drop = nn.Dropout(proj_drop)
        self.apply(self._init_weights)
        self.insert_cls_layer = insert_cls_layer

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, e):
        N, C, T, V = x.shape
        kv = self.kv(x).reshape(N, 2, self.num_heads, self.dim // self.num_heads, T, V).permute(1, 0, 4, 2, 5, 3)
        k, v = kv[0], kv[1]

        ## n t h v c
        q = self.q(x).reshape(N, self.num_heads, self.dim // self.num_heads, T, V).permute(0, 3, 1, 4, 2)

        e_k = e.reshape(N, self.num_heads, self.dim // self.num_heads, T, V).permute(0, 3, 1, 4, 2)
        #
        #
        pos_emb = self.rpe[self.hops]
        #
        k_r = pos_emb.view(V, V, self.num_heads, self.dim // self.num_heads)
        #
        b = torch.einsum("bthnc, nmhc->bthnm", q, k_r)
        #
        c = torch.einsum("bthnc, bthmc->bthnm", q, e_k)
        d = torch.einsum("hc, bthmc->bthm", self.w1, e_k).unsqueeze(-2)


        a = q @ k.transpose(-2, -1)

        attn = a + b + c + d


        attn = attn * self.scale

        attn = attn.softmax(dim=-1)

        attn = self.attn_drop(attn)


        x = (self.alpha * attn + self.outer) @ v
        # x = attn @ v


        x = x.transpose(3, 4).reshape(N, T, -1, V).transpose(1, 2)
        x = self.proj(x)

        x = self.proj_drop(x)
        return x

# using conv2d implementation after dimension permutation
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.,
                 num_heads=None):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1)
        self.drop = nn.Dropout(drop)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        # x = self.fc1(x.transpose(1,2)).transpose(1,2)
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        # x = self.fc2(x.transpose(1,2)).transpose(1,2)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class unit_vit(nn.Module):
    def __init__(self, dim_in, dim, A, num_of_heads, add_skip_connection=True,  qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0, act_layer=nn.GELU, norm_layer=nn.LayerNorm, layer=0,
                insert_cls_layer=0, pe=False, num_point=25, **kwargs):
        super().__init__()
        self.norm1 = norm_layer(dim_in)
        self.dim_in = dim_in
        self.dim = dim
        self.add_skip_connection = add_skip_connection
        self.num_point = num_point
        self.attn = MHSA(dim_in, dim, A, num_heads=num_of_heads, qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop,
                             proj_drop=drop, insert_cls_layer=insert_cls_layer, pe=pe, num_point=num_point, layer=layer, **kwargs)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        if self.dim_in != self.dim:
            self.skip_proj = nn.Conv2d(dim_in, dim, (1, 1), padding=(0, 0), bias=False)
        self.pe_proj = nn.Conv2d(dim_in, dim, 1, bias=False)
        self.pe = pe

    def forward(self, x, joint_label, groups):
        ## more efficient implementation
        label = F.one_hot(torch.tensor(joint_label)).float().to(x.device)
        z = x @ (label / label.sum(dim=0, keepdim=True))

        # w/o proj
        # z = z.permute(3, 0, 1, 2)
        # w/ proj
        z = self.pe_proj(z).permute(3, 0, 1, 2)

        e = z[joint_label].permute(1, 2, 3, 0)

        if self.add_skip_connection:
            if self.dim_in != self.dim:
                x = self.skip_proj(x) + self.drop_path(self.attn(self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2), e))
            else:
                x = x + self.drop_path(self.attn(self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2), e))
        else:
            x = self.drop_path(self.attn(self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2), e))

        # x = x + self.drop_path(self.mlp(self.norm2(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)))

        return x

class TCN_ViT_unit(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, num_of_heads=6, residual=True, kernel_size=5, dilations=[1,2], pe=False, num_point=25, layer=0):
        super(TCN_ViT_unit, self).__init__()
        self.vit1 = unit_vit(in_channels, out_channels, A, add_skip_connection=residual, num_of_heads=num_of_heads, pe=pe, num_point=num_point, layer=layer)
        # self.tcn1 = unit_tcn(out_channels, out_channels, stride=stride)
        self.tcn1 = MultiScale_TemporalConv(out_channels, out_channels, kernel_size=kernel_size, stride=stride,
                                            dilations=dilations,
                                            # redisual=True has worse performance in the end
                                            residual=False)
        self.act = nn.ReLU(inplace=True)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

        if not residual:
            self.residual = lambda x: 0

        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x

        else:
            self.residual = unit_tcn(in_channels, out_channels, kernel_size=1, stride=stride)

    def forward(self, x, joint_label, groups):
        y = self.act(self.tcn1(self.vit1(x, joint_label, groups)) + self.residual(x))
        return y


class Model(nn.Module):
    def __init__(self, num_class=60, num_point=20, num_person=2, graph=None, graph_args=dict(), in_channels=3,
                 drop_out=0, num_of_heads=9, joint_label=[], **kwargs):
        super(Model, self).__init__()

        self_link = [(i, i) for i in range(25)]
        inward_ori_index = [(1, 2), (2, 21), (3, 21), (4, 3), (5, 21), (6, 5), (7, 6),
                    (8, 7), (9, 21), (10, 9), (11, 10), (12, 11), (13, 1),
                    (14, 13), (15, 14), (16, 15), (17, 1), (18, 17), (19, 18),
                    (20, 19), (22, 23), (23, 8), (24, 25), (25, 12)]
        inward = [(i - 1, j - 1) for (i, j) in inward_ori_index]
        outward = [(j, i) for (i, j) in inward]
        self.neighbor = inward + outward
        A = self.get_spatial_graph(25, self_link, inward, outward)

        self.num_of_heads = num_of_heads
        self.num_class = num_class
        self.num_point = num_point
        self.num_person = num_person
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)
        self.joint_label = joint_label


        self.l1 = TCN_ViT_unit(3, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=1)
        # * num_heads, effect of concatenation following the official implementation
        self.l2 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=2)
        self.l3 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=3)
        self.l4 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=4)
        self.l5 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, stride=2, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=5)
        self.l6 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=6)
        self.l7 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=7)
        # self.l8 = TCN_ViT_unit(24 * num_of_heads, 24 * num_of_heads, A, num_of_heads=num_of_heads)
        self.l8 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, stride=2, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=8)
        self.l9 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=9)
        self.l10 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=10)
        # standard ce loss
        self.fc = nn.Linear(24*num_of_heads, num_class)
        
        # ## larger model
        # self.l1 = TCN_ViT_unit(3, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True,
        #                        num_point=num_point, layer=1)
        # # * num_heads, effect of concatenation following the official implementation
        # self.l2 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=2)
        # self.l3 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=3)
        # self.l4 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=4)
        # self.l5 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, stride=2,
        #                        num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=5)
        # self.l6 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=6)
        # self.l7 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=7)
        # # self.l8 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, num_of_heads=num_of_heads)
        # self.l8 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, stride=2,
        #                        num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=8)
        # self.l9 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                        pe=True, num_point=num_point, layer=9)
        # self.l10 = TCN_ViT_unit(36 * num_of_heads, 36 * num_of_heads, A, residual=True, num_of_heads=num_of_heads,
        #                         pe=True, num_point=num_point, layer=10)
        # # standard ce loss
        # self.fc = nn.Linear(36 * num_of_heads, num_class)

        nn.init.normal_(self.fc.weight, 0, math.sqrt(2. / num_class))
        bn_init(self.data_bn, 1)
        if drop_out:
            self.drop_out = nn.Dropout(drop_out)
        else:
            self.drop_out = lambda x: x

    def forward(self, x, y):
        groups = []
        for num in range(max(self.joint_label)+1):
            groups.append([ind for ind, element in enumerate(self.joint_label) if element==num])

        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)

        ## n, c, t, v
        x = x.view(N, M, V, C, T).contiguous().view(N * M, V, C, T).permute(0, 2, 3, 1)

        x = self.l1(x, self.joint_label, groups)
        x = self.l2(x, self.joint_label, groups)
        x = self.l3(x, self.joint_label, groups)
        x = self.l4(x, self.joint_label, groups)
        x = self.l5(x, self.joint_label, groups)
        x = self.l6(x, self.joint_label, groups)
        x = self.l7(x, self.joint_label, groups)
        x = self.l8(x, self.joint_label, groups)
        x = self.l9(x, self.joint_label, groups)
        x = self.l10(x, self.joint_label, groups)

        # N*M, C, T, V
        _ , C, T, V = x.size()
        # spatial temporal average pooling
        x = x.view(N, M, C, -1)
        x = x.mean(3).mean(1)
        x = self.drop_out(x)
        x = self.fc(x)

        return x, y

    
    def edge2mat(self, link, num_node):
        A = np.zeros((num_node, num_node))
        for i, j in link:
            A[j, i] = 1
        return A
    
    def normalize_digraph(self, A):
        Dl = np.sum(A, 0)
        h, w = A.shape
        Dn = np.zeros((w, w))
        for i in range(w):
            if Dl[i] > 0:
                Dn[i, i] = Dl[i] ** (-1)
        AD = np.dot(A, Dn)
        return AD
    
    def get_spatial_graph(self, num_node, self_link, inward, outward):
        I = self.edge2mat(self_link, num_node)
        In = self.normalize_digraph(self.edge2mat(inward, num_node))
        Out = self.normalize_digraph(self.edge2mat(outward, num_node))
        A = np.stack((I, In, Out))
        return A



In [ ]:
def train(epoch, batch_size):
    model.train()
    loss_value = []
    acc_value = []
    train_loader = load_data(phase='train', batch_size=batch_size)
    process = tqdm(train_loader, ncols=80)
    for batch_idx, (data, label, index) in enumerate(process):
        with torch.no_grad():
            data = data.float().to(device)
            label = label.long().to(device)
        out, z = model(data, F.one_hot(label, num_classes=60))
        loss = lossC(out, label)
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping
        #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # Step the optimizer
        optimizer.step()
        loss_value.append(loss.data.item())
        value, predict_label = torch.max(out.data, 1)
        acc = torch.mean((predict_label == label.data).float())
        acc_value.append(acc.data.item())
    return np.nanmean(loss_value), np.nanmean(acc_value)*100

In [ ]:
def evalt(batch_size, wrong_file = None, result_file = None, model_name = 'Model', save=True, loader='S'):
    loader = load_data(phase='test', batch_size=batch_size, data=loader)
    if wrong_file is not None:
            f_w = open(wrong_file, 'w')
    if result_file is not None:
        f_r = open(result_file, 'w')
    model.eval()
    #print('Eval epoch: {}'.format(epoch + 1))
    loss_value = []
    score_frag = []
    show_topk = [1, 5]
    process = tqdm(loader, ncols=80)
    for batch_idx, (data, label, index) in enumerate(process):
        with torch.no_grad():
            data = data.float().cuda(device)
            label = label.long().cuda(device)
            output, z = model(data, F.one_hot(label, num_classes=60))

            loss = lossC(output, label)

            score_frag.append(output.data.cpu().numpy())
            loss_value.append(loss.data.item())

            _, predict_label = torch.max(output.data, 1)

        if wrong_file is not None or result_file is not None:
            predict = list(predict_label.cpu().numpy())
            true = list(label.data.cpu().numpy())
            for i, x in enumerate(predict):
                if result_file is not None:
                    f_r.write(str(x) + ',' + str(true[i]) + '\n')
                if x != true[i] and wrong_file is not None:
                    f_w.write(str(index[i]) + ',' + str(x) + ',' + str(true[i]) + '\n')

    score = np.concatenate(score_frag)
    loss = np.mean(loss_value)
    accuracy = loader.dataset.top_k(score, 1)
    print('Accuracy: ', accuracy, ' model: ', model_name)
    print('\tMean {} loss of {} batches: {}.'.format(
        "test", len(loader), np.mean(loss_value)))
    for k in show_topk:
        print('\tTop{}: {:.2f}%'.format(
            k, 100 * loader.dataset.top_k(score, k)))

    top5_acc = loader.dataset.top_k(score, 5) * 100
    top1_acc = loader.dataset.top_k(score, 1) * 100
    global best_epoch_acc
    if save and top1_acc > best_epoch_acc:
        state_dict = model.state_dict()
        torch.save(state_dict, f'mainruns/{model_name}.pt')
        best_epoch_acc = top1_acc
    return top5_acc, loss, top1_acc

In [ ]:
def load_data( phase='train', batch_size=32, data='S'):
    if phase=='train':
        data_loader = torch.utils.data.DataLoader(
            dataset=Feeder(f'data/ntu/NTU60_C{data}.npz', split='train', p_interval =[0.5, 1], window_size=64),
            batch_size=batch_size,
            num_workers=24,
            prefetch_factor=16,
            shuffle=True,
            pin_memory=True,
            drop_last=True,
            worker_init_fn=init_seed)
    else:
        data_loader = torch.utils.data.DataLoader(
            dataset=Feeder(f'data/ntu/NTU60_C{data}.npz', split='test', p_interval =[0.95], window_size=64),
            batch_size=128,
            num_workers=24,
            shuffle=False,
            drop_last=False,
            worker_init_fn=init_seed)
    return data_loader

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

def adjust_learning_rate(epoch):
    global lr, warm_epochs, lr_decay_rate, step
    if epoch <= warm_epochs:
        lr = base_lr * epoch / warm_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    else:
        if sched == 'plat':
            scheduler.step(tst[1])
        elif sched == 'cos':
            scheduler.step()
        elif sched == 'custom':
            lr = base_lr * (lr_decay_rate ** np.sum(epoch >= np.array(step)))
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
    return lr

In [ ]:
def init_seed(seed):
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # torch.backends.cudnn.enabled = True
    # training speed is too slow if set to True
    torch.backends.cudnn.deterministic = True

    # on cuda 11 cudnn8, the default algorithm is very slow
    # unlike on cuda 10, the default works well
    torch.backends.cudnn.benchmark = False

In [ ]:
init_seed(2)

torch.cuda.empty_cache()
#torch.autograd.set_detect_anomaly(True)
#import os
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

phase = 'train'
optimz = 'SGD'
warm_epochs = 5
epochs = 140
batch_size = 64
out_channels = 160
weight_decay = 0.0004
if optimz == 'Adam' or optimz == 'AdamW' or optimz == 'NAdam':
    base_lr = 0.001
    lr = base_lr
    lr_decay_rate = 0.1
    step = [30, 60]
    sched = 'cos'
elif optimz == 'SGD':
    base_lr = 0.025
    lr = base_lr
    lr_decay_rate = 0.1
    step = [110, 120]
    sched = 'custom'

use_wandb = True
'''device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
model = Model(num_class=60, num_point=25, num_person=2, in_channels=3,
                 drop_out=0, num_of_heads=9, joint_label=[0, 4, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 2, 3, 3, 3, 2, 3, 3, 3, 1, 0, 1, 0, 1]).to(device)'''
# model.apply(init_weights)
devices = [2, 0]
device = devices[0]
model = Model(num_class=60, num_point=25, num_person=2, in_channels=3,
                 drop_out=0, num_of_heads=9, joint_label=[0, 4, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 2, 3, 3, 3, 2, 3, 3, 3, 1, 0, 1, 0, 1]).to(device)
model = nn.DataParallel(model,
                        device_ids=devices,
                        output_device=device)
model.to(device)
#model = nn.SyncBatchNorm.convert_sync_batchnorm(model)

if optimz == 'SGD':
    optimizer = optim.SGD(
                model.parameters(),
                momentum=0.9,
                lr=base_lr,
                nesterov = True,
                weight_decay = weight_decay)
elif optimz == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=base_lr)
elif optimz == 'NAdam':
    optimizer = optim.NAdam(model.parameters(), lr=base_lr, weight_decay=weight_decay)
elif optimz == 'AdamW':
    optimizer = optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
'''for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    params = parameter.numel()
    print(name, params)'''
lossC = nn.CrossEntropyLoss().to(device)
# scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
if sched == 'plat':
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=lr_decay_rate, patience=5)
elif sched == 'cos':
    scheduler = CosineAnnealingLR(optimizer, epochs, base_lr*lr_decay_rate)
if phase == 'train':
    if use_wandb:
        wandb.login()
        wandb.init(
            # set the wandb project where this run will be logged
            project="spanchsan-hong-kong-polytechnic-university",
        
            # track hyperparameters and run metadata
            config={
            "learning_rate": lr,
            "architecture": "Mamba",
            "dataset": "NTU RGB+D 60",
            "epochs": epochs,
            "out_channels": out_channels,
            "batch_size": batch_size,
            "weight_decay": weight_decay,
            "lr_decay_rate": lr_decay_rate,
            "optimizer": optimz,
            "scheduler": sched
            }
        )
    print("Parameters:", count_parameters(model))
    print(model)
    best_epoch_acc = 0
    for epoch in range(1, epochs+1):
        adjust_learning_rate(epoch)
        trn = train(epoch, batch_size)
        if use_wandb:
            tst = evalt(batch_size, model_name = f'Model{"Hyperformer"}_{wandb.run.name}')
        else:
            tst = evalt(batch_size, save = False)
        if use_wandb:
            wandb.log({"train_loss": trn[0], "train_acc": trn[1], "test_acc": tst[0], "test_acc_1st": tst[2], "test_loss": tst[1]})
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch: {epoch:02d}, Loss: {trn[0]:.4f}, Train Acc: {trn[1]:.4f}, \
        Accuracy: {tst[2]:.4f}, Test_Loss: {tst[1]:.4f}, Current_Lr: {current_lr:.6f}')
elif phase == 'test':
    print("Parameters:", count_parameters(model))
    weights_path = 'mainruns/Model192_cherubic-violet-165.pt'
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    wf = weights_path.replace('.pt', '_wrong.txt')
    rf = weights_path.replace('.pt', '_right.txt')
    evalt(batch_size = 32, wrong_file=wf, result_file=rf, save=False, loader='S')